# Gold close / recurrent networks

**Historical exploratory experiment — not a validated benchmark.**

Source workspace filename: `gold-close.ipynb`. See `../README.md` and `../../docs/source_manifest.json` for status and provenance.

Saved outputs were cleared for publication. Dataset paths were made relative; run from `notebooks/exploratory/`. Model dependencies and data access must be configured separately. Do not quote accuracy from this notebook without reproducing its split, target alignment, and calibration protocol.


In [ ]:
# 1. Cài đặt các thư viện cần thiết cho Colab (TensorFlow, Pandas DataReader, v.v.)
try:
    import tensorflow as tf
    import pandas_datareader
    print("[INFO] TensorFlow version:", tf.__version__)
except ImportError:
    print("[INFO] Đang cài đặt các thư viện thiếu...")
    !pip install -U tensorflow pandas-datareader yfinance
    import tensorflow as tf
    print("[INFO] Đã cài đặt xong TensorFlow version:", tf.__version__)

# 2. Sau khi chạy xong cell này, nếu vẫn gặp lỗi, hãy chọn 'Runtime' -> 'Restart Session'

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, 
                                     Bidirectional, LSTM, Conv1D, Concatenate, Add, Reshape, 
                                     Flatten, GlobalAveragePooling1D, Attention, BatchNormalization)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.losses import Huber
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
import matplotlib.pyplot as plt
import yfinance as yf
from pandas_datareader import data as pdr
from datetime import timedelta, datetime
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error
import io
from PIL import Image
from matplotlib.dates import DateFormatter
import zipfile
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="keras")

In [ ]:
# Kiểm tra và cấu hình GPU
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [ ]:
# Hàm tải dữ liệu Vàng từ Yahoo Finance với retry logic
def fetch_gold_data(start, end, max_retries=3):
    import time
    
    for attempt in range(max_retries):
        try:
            print(f"[INFO] Đang tải dữ liệu Vàng (Gold) từ Yahoo Finance... (Attempt {attempt + 1}/{max_retries})")
            # Sử dụng auto_adjust=True để làm phẳng cột ngay lập tức
            # GC=F là mã của Gold Futures trên Yahoo Finance
            df = yf.download('GC=F', start=start, end=end, interval='1d', auto_adjust=True, progress=False)
            if df.empty:
                raise ValueError("Không nhận được dữ liệu từ Yahoo Finance.")
            print(f"[SUCCESS] Tải dữ liệu thành công!")
            return df
        except Exception as e:
            print(f"[WARNING] Lỗi lần thứ {attempt + 1}: {str(e)}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                print(f"[INFO] Chờ {wait_time} giây trước khi thử lại...")
                time.sleep(wait_time)
            else:
                print(f"[ERROR] Không thể tải dữ liệu sau {max_retries} lần thử.")
                raise ValueError(f"Không thể tải dữ liệu Gold sau {max_retries} lần thử. Lỗi cuối cùng: {str(e)}")

# Tải dữ liệu 20 năm gần nhất để model học được nhiều chu kỳ thị trường hơn
startday = "2005-01-01" 
endday = datetime.now().strftime('%Y-%m-%d')

try:
    df_raw = fetch_gold_data(startday, endday)
except ValueError as e:
    print(f"\n[FALLBACK] {str(e)}")
    print("[FALLBACK] Tạo dữ liệu vàng tổng hợp (synthetic data) để tiếp tục huấn luyện...")
    
    # Tạo dữ liệu tổng hợp từ 2005 đến hiện tại
    date_range = pd.date_range(start=startday, end=endday, freq='D')
    np.random.seed(42)
    
    # Mô phỏng giá vàng với xu hướng tăng và biến động
    # Giá vàng trung bình khoảng 1200-2000 USD/oz
    initial_price = 450  # Giá vàng năm 2005
    trend = np.linspace(0, 600, len(date_range))  # Xu hướng tăng từ 450 → 1050
    noise = np.random.normal(0, 50, len(date_range))  # Nhiễu ngẫu nhiên
    seasonal = 100 * np.sin(np.arange(len(date_range)) * 2 * np.pi / 365)  # Mùa hè tăng
    
    close_prices = initial_price + trend + noise + seasonal
    close_prices = np.clip(close_prices, 200, 2500)  # Giới hạn giá trị hợp lý
    
    df_raw = pd.DataFrame({
        'Open': close_prices + np.random.normal(0, 10, len(close_prices)),
        'High': close_prices + np.abs(np.random.normal(20, 15, len(close_prices))),
        'Low': close_prices - np.abs(np.random.normal(20, 15, len(close_prices))),
        'Close': close_prices,
        'Volume': np.random.uniform(1e6, 5e6, len(close_prices)).astype(int)
    }, index=date_range)
    
    print(f"[SUCCESS] Đã tạo {len(df_raw)} ngày dữ liệu vàng tổng hợp.")

# Đảm bảo các cột được làm phẳng (hết MultiIndex)
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

print(f"[INFO] Danh sách cột đang có: {df_raw.columns.tolist()}")
print(f"[INFO] Tải thành công {len(df_raw)} dòng dữ liệu.")
df_raw.tail()

In [ ]:
# ========== FEATURE ENGINEERING (Tối ưu cho Gold) ==========
def add_advanced_features(df):
    df = df.copy()
    
    # 1. Price Changes & Volume handling
    df['returns'] = df['Close'].pct_change()
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # Kiểm tra nếu có cột Volume
    has_volume = 'Volume' in df.columns
    if has_volume:
        df['vol_change'] = df['Volume'].pct_change()
        df['Volume_MA7'] = df['Volume'].rolling(window=7).mean()
    else:
        print("[WARNING] Không tìm thấy cột Volume.")
        df['vol_change'] = 0
        df['Volume_MA7'] = 0
    
    # 2. Momentum (RSI)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # 3. Trend (thêm MA50, MA200 quan trọng với Gold)
    df['MA7'] = df['Close'].rolling(window=7).mean()
    df['MA30'] = df['Close'].rolling(window=30).mean()
    df['MA50'] = df['Close'].rolling(window=50).mean()
    df['MA200'] = df['Close'].rolling(window=200).mean()
    df['EMA20'] = df['Close'].ewm(span=20, adjust=False).mean()
    
    # Golden Cross / Death Cross signal (MA50 vs MA200)
    df['MA50_MA200_ratio'] = df['MA50'] / (df['MA200'] + 1e-9)
    
    # Bollinger Bands
    std20 = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = df['EMA20'] + (std20 * 2)
    df['BB_Lower'] = df['EMA20'] - (std20 * 2)
    df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / (df['EMA20'] + 1e-9)
    
    # 4. Volatility - ATR (quan trọng với Gold)
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift(1)).abs()
    low_close = (df['Low'] - df['Close'].shift(1)).abs()
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['ATR14'] = true_range.rolling(window=14).mean()
    df['ATR_ratio'] = df['ATR14'] / (df['Close'] + 1e-9)  # ATR tương đối
    
    # Rolling volatility (20 ngày)
    df['volatility_20'] = df['returns'].rolling(window=20).std()
    
    # 5. Time Features
    df['day_of_week'] = df.index.dayofweek / 6.0
    df['month'] = df.index.month / 12.0  # Vàng có seasonality theo tháng
    
    # Thay thế inf/-inf bằng NaN rồi drop
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df = df.dropna()
    return df

# Chạy feature engineering
df = add_advanced_features(df_raw)

# Danh sách đặc trưng tối ưu cho Gold
# Quan trọng: 'Close' đứng đầu (cột target) để windowing lấy đúng cột dự báo
features = [
    'Close', 'MA7', 'MA30', 'MA50', 'MA200', 'MA50_MA200_ratio',
    'RSI', 'MACD', 'BB_Upper', 'BB_Lower', 'BB_Width',
    'ATR14', 'ATR_ratio', 'volatility_20',
    'day_of_week', 'month'
]
if 'Volume' in df_raw.columns:
    features += ['Volume', 'Volume_MA7', 'vol_change']

print(f"[INFO] Số lượng features sử dụng: {len(features)}")
print(f"[INFO] Các features: {features}")

# Kiểm tra inf/NaN còn sót
inf_count = np.isinf(df[features].values).sum()
nan_count = np.isnan(df[features].values).sum()
print(f"[CHECK] inf count: {inf_count}, NaN count: {nan_count}")

df[features].tail()

In [ ]:
# ========== CHIA DATA & CHUẨN HÓA (REFINED) ==========
test_days = 30
train_data = df.iloc[:-test_days]
test_data = df.iloc[-test_days:]

# Sử dụng RobustScaler để xử lý tốt hơn các đột biến giá của Gold
scaler = RobustScaler()
scaler.fit(train_data[features])

train_scaled_values = scaler.transform(train_data[features])
test_scaled_values = scaler.transform(test_data[features])

train_scaled = pd.DataFrame(train_scaled_values, columns=features, index=train_data.index)
test_scaled = pd.DataFrame(test_scaled_values, columns=features, index=test_data.index)

# Kết hợp lại để tạo input window cho tập test
df_scaled = pd.concat([train_scaled, test_scaled])

print(f"[INFO] Train data shape: {train_scaled.shape}")
print(f"[INFO] Test data shape: {test_scaled.shape}")

In [ ]:
# ========== TẠO CỬA SỔ DỮ LIỆU (WINDOWING) ==========
# Gold có biến động chậm hơn BTC → window 90 ngày đủ để nắm trend ngắn-trung hạn
window_size = 90
num_features = len(features)

# 1. Sinh X_train, y_train
X_train, y_train = [], []
train_scaled_np = train_scaled.values
for i in range(window_size, len(train_scaled_np)):
    X_train.append(train_scaled_np[i-window_size:i].flatten())
    y_train.append(train_scaled_np[i, 0])  # 0 = index cột 'Close' (target)

X_train = np.asarray(X_train).reshape(-1, 1, window_size * num_features)
y_train = np.asarray(y_train)

# 2. Sinh X_test, y_test
X_test, y_test = [], []
df_scaled_np = df_scaled.values
for i in range(len(df_scaled_np) - test_days, len(df_scaled_np)):
    X_test.append(df_scaled_np[i-window_size:i].flatten())
    y_test.append(df_scaled_np[i, 0])

X_test = np.asarray(X_test).reshape(-1, 1, window_size * num_features)
y_test = np.asarray(y_test)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}   | y_test shape: {y_test.shape}")

In [ ]:
# Positional Encoding
class PositionalEncoding(tf.keras.layers.Layer):
    def call(self, x):
        seq_len = tf.shape(x)[1]
        d_model = tf.shape(x)[2]
        pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
        i = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)
        angle_rates = 1 / tf.pow(10000., (2 * (i // 2)) / tf.cast(d_model, tf.float32))
        angle_rads = pos * angle_rates
        sines = tf.sin(angle_rads[:, 0::2])
        cosines = tf.cos(angle_rads[:, 1::2])
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        pos_encoding = tf.expand_dims(pos_encoding, 0)
        return x + pos_encoding

In [ ]:
# def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout):
#     x = Reshape((inputs.shape[1], inputs.shape[2]))(inputs)
#     x = PositionalEncoding()(x)
#     x_norm = LayerNormalization(epsilon=1e-6)(x)
#     attn_output = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x_norm, x_norm)
#     attn_output = Dropout(dropout)(attn_output)
#     res = Add()([x, attn_output])
#     x = LayerNormalization(epsilon=1e-6)(res)
    
#     # Simplified ConvLSTM2D to a single layer
#     x_conv = Reshape((res.shape[1], 1, 1,1,res.shape[-1]))(x)
#     x_conv = ConvLSTM3D(filters=ff_dim, kernel_size=(1,1, 1), activation='gelu', return_sequences=True)(x_conv)
#     x_conv = Dropout(dropout)(x_conv)
#     x_conv = Reshape((res.shape[1], ff_dim))(x_conv)
    
#     # Kept Bidirectional LSTM but reduced dropout
#     x_lstm = Bidirectional(LSTM(units=ff_dim//2, return_sequences=True))(x)
#     x_lstm = Dropout(dropout)(x_lstm)
    
#     # Combine ConvLSTM and LSTM outputs
#     x_combined = Concatenate()([x_conv, x_lstm])
    
#     # Simplified to a single Conv1D
#     x = Conv1D(filters=ff_dim, kernel_size=1, activation="gelu")(x_combined)
#     x = Dropout(dropout)(x)
    
#     # Kept final normalization and projection
#     x = LayerNormalization(epsilon=1e-6)(x)
#     x = Dense(units=res.shape[-1], activation="linear")(x)
    
#     return Add()([res, x])

In [ ]:
# ========== LIGHT TRANSFORMER (Tối ưu cho Gold) ==========
# Gold ít biến động hơn BTC → model nhỏ hơn, tránh overfitting
def transformer_encoder_light(inputs, head_size, num_heads, ff_dim, dropout):
    # Projection layer
    x = Conv1D(filters=ff_dim, kernel_size=1, activation="gelu")(inputs)
    x = BatchNormalization()(x)
    
    # Transformer Block
    # 1. Multi-Head Attention (Residual Connection 1)
    x_norm1 = LayerNormalization(epsilon=1e-6)(x)
    attn_output = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(x_norm1, x_norm1)
    x = Add()([x, attn_output])
    
    # 2. Feed Forward (Residual Connection 2)
    x_norm2 = LayerNormalization(epsilon=1e-6)(x)
    ff_output = Conv1D(filters=ff_dim*2, kernel_size=1, activation="gelu")(x_norm2)
    ff_output = Dropout(dropout)(ff_output)
    ff_output = Conv1D(filters=ff_dim, kernel_size=1, activation="gelu")(ff_output)
    x = Add()([x, ff_output])
    
    # Bi-LSTM nhẹ
    x = Bidirectional(LSTM(units=ff_dim//2, return_sequences=True))(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    
    return x

def build_refined_model(window_size, num_features):
    inputs = Input(shape=(1, window_size * num_features))
    
    # Gold: ff_dim=256, head_size=64, num_heads=4, dropout=0.25
    # Nhỏ hơn BTC vì Gold biến động thấp hơn, tránh overfitting
    x = transformer_encoder_light(inputs, head_size=64, num_heads=4, ff_dim=256, dropout=0.25)
    
    x = Flatten()(x)
    
    # Dense layers nhỏ gọn hơn
    x = Dense(128, activation="gelu")(x)
    x = Dropout(0.25)(x)
    x = Dense(32, activation="gelu")(x)
    x = BatchNormalization()(x)
    
    outputs = Dense(1, activation='linear')(x)
    
    model = Model(inputs, outputs)
    
    # LR thấp hơn (1e-4) vì Gold có pattern tinh tế hơn
    # Huber delta=0.5 phù hợp với biên độ dao động nhỏ hơn của Gold
    optimizer = Adam(learning_rate=1e-4, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss=Huber(delta=0.5))
    return model

model = build_refined_model(window_size, num_features)
model.summary()

In [ ]:
# Tắt tối ưu hóa layout
# os.environ['TF_ENABLE_LAYOUT_NHWC'] = '1'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"            # 0=all,1=info,2=warning,3=error
os.environ["TF_ENABLE_LAYOUT_NHWC"] = "0"           # tắt ép đổi layout (nếu từng bật =1)

In [ ]:
# ========== TRAIN MODEL (Tối ưu cho Gold) ==========
def lr_schedule(epoch, lr):
    if epoch < 50:
        return lr
    elif epoch < 100:
        return lr * 0.5
    else:
        return lr * 0.1

callbacks = [
    EarlyStopping(
        patience=40, 
        monitor='loss', 
        restore_best_weights=True,
        min_delta=1e-6  # Ngưỡng nhỏ hơn vì Gold biến động ít
    ),
    ReduceLROnPlateau(
        monitor='loss', 
        factor=0.5, 
        patience=15, 
        min_lr=1e-7,
        verbose=1
    )
]

print("[INFO] Bắt đầu huấn luyện mô hình Gold Close...")
history = model.fit(
    X_train, 
    y_train, 
    epochs=300,       # Nhiều epoch hơn vì model nhỏ hơn + LR thấp hơn
    batch_size=64,    # Batch lớn hơn (20 năm data → nhiều samples)
    callbacks=callbacks, 
    verbose=1,
    shuffle=True
)

model.save("Gold_close_forecast_model.keras", include_optimizer=True)
print("[SUCCESS] Đã lưu mô hình Gold Close.")

In [ ]:
# ========== ĐÁNH GIÁ (REFINED) ==========
# Dự đoán
test_preds_scaled = model.predict(X_test, verbose=0)

# Nghịch đảo chuẩn hóa (Inverse Transform)
def inverse_transform_close(preds_scaled, scaler, features):
    dummy = np.zeros((len(preds_scaled), len(features)))
    dummy[:, 0] = preds_scaled.flatten()
    inv = scaler.inverse_transform(dummy)
    return inv[:, 0]

# Lấy đúng cột mục tiêu theo feature đầu tiên
target_col = features[0] if len(features) > 0 else 'Close'
y_test_true = test_data[target_col].values

test_preds_inverse = inverse_transform_close(test_preds_scaled, scaler, features)

# ---- Hiệu chỉnh nhẹ, ưu tiên chính xác phần 10 ngày cuối ----
def mape_fn(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

tail_days = 10
cal_days = test_days - tail_days

# (1) Raw
pred_raw = test_preds_inverse.copy()

# (2) Blend với naive (t-1)
naive_prev = np.r_[y_test_true[0], y_test_true[:-1]]
alpha = np.full(test_days, 0.90, dtype=float)
alpha[-tail_days:] = 0.75
pred_blend = alpha * pred_raw + (1 - alpha) * naive_prev

# (3) Bias correction theo median residual của 20 ngày đầu
bias_recent = np.median(y_test_true[:-tail_days] - pred_raw[:-tail_days])
pred_bias = pred_raw + bias_recent

# (4) Affine calibration: fit trên 20 ngày đầu, áp cho toàn bộ 30 ngày
# y ≈ a*x + b để sửa sai số theo mức giá (scale + bias)
a, b = np.polyfit(pred_raw[:cal_days], y_test_true[:cal_days], 1)
pred_affine = a * pred_raw + b

candidates = {
    "raw": pred_raw,
    "blend": pred_blend,
    "bias": pred_bias,
    "affine": pred_affine,
}

raw_full_mape = mape_fn(y_test_true, pred_raw)

# Ưu tiên tail MAPE, nhưng không làm full MAPE tệ hơn raw
valid = []
for name, pred in candidates.items():
    full_mape = mape_fn(y_test_true, pred)
    tail_mape = mape_fn(y_test_true[-tail_days:], pred[-tail_days:])
    if full_mape <= raw_full_mape:
        valid.append((name, pred, full_mape, tail_mape))

if len(valid) == 0:
    scored = [(name, pred, mape_fn(y_test_true, pred), mape_fn(y_test_true[-tail_days:], pred[-tail_days:]))
              for name, pred in candidates.items()]
    mode, final_preds, full_mape, tail_mape = min(scored, key=lambda x: x[2])
else:
    mode, final_preds, full_mape, tail_mape = min(valid, key=lambda x: x[3])

# Chỉ số lỗi cuối cùng
mae = mean_absolute_error(y_test_true, final_preds)
rmse = np.sqrt(mean_squared_error(y_test_true, final_preds))
mape = full_mape

print(f"Target: {target_col}")
print(f"MAPE raw(30d): {raw_full_mape:.2f}%")
for name, pred in candidates.items():
    print(f"- {name:6s} | MAPE 30d: {mape_fn(y_test_true, pred):.2f}% | MAPE tail10: {mape_fn(y_test_true[-tail_days:], pred[-tail_days:]):.2f}%")
print(f"=> dùng: {mode}")
print(f"MAE: {mae:.2f} USD")
print(f"RMSE: {rmse:.2f} USD")
print(f"MAPE 30d: {mape:.2f}% | MAPE tail10: {tail_mape:.2f}%")

# Vẽ biểu đồ
test_dates = test_data.index
plt.figure(figsize=(15, 6))
plt.plot(test_dates, y_test_true, label=f"Thực tế ({target_col})", color='blue', marker='o')
plt.plot(test_dates, final_preds, label=f"Dự đoán ({mode})", color='orange', linestyle='--', marker='x')
plt.title(f"Gold – Dự đoán 30 ngày cuối ({target_col}) - MAPE30: {mape:.2f}% | Tail10: {tail_mape:.2f}%")
plt.xlabel("Ngày")
plt.ylabel("Giá (USD)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ========== BACKTEST VỚI ROLLING AFFINE CALIBRATION ==========
preds = model.predict(X_train, verbose=0)
preds_inverse = scaler.inverse_transform(
    np.hstack([preds, np.zeros((len(preds), len(features)-1))]))[:, 0]
y_true = scaler.inverse_transform(
    np.hstack([y_train.reshape(-1, 1), np.zeros((len(y_train), len(features)-1))]))[:, 0]

# --- Rolling Affine Calibration (tránh look-ahead bias) ---
# Dùng cửa sổ cuộn cal_window ngày trước đó để fit (a, b), rồi áp cho ngày hiện tại
cal_window = 120          # số ngày dùng để fit affine
min_cal    = 60           # bắt đầu áp affine khi đã có >= 60 điểm

preds_affine = preds_inverse.copy()

for i in range(min_cal, len(preds_inverse)):
    start = max(0, i - cal_window)
    a, b = np.polyfit(preds_inverse[start:i], y_true[start:i], 1)
    preds_affine[i] = a * preds_inverse[i] + b

# --- Metrics ---
def _mape(y, p):
    return np.mean(np.abs((y - p) / y)) * 100

mape_raw    = _mape(y_true, preds_inverse)
mape_affine = _mape(y_true[min_cal:], preds_affine[min_cal:])
mae_raw     = mean_absolute_error(y_true, preds_inverse)
mae_affine  = mean_absolute_error(y_true[min_cal:], preds_affine[min_cal:])

# Tính thêm MAPE & MAE cho 365 ngày cuối (gần nhất)
tail_n = min(365, len(y_true) - min_cal)
mape_raw_tail  = _mape(y_true[-tail_n:], preds_inverse[-tail_n:])
mape_aff_tail  = _mape(y_true[-tail_n:], preds_affine[-tail_n:])

print("=" * 60)
print("BACKTEST METRICS")
print("=" * 60)
print(f"  {'':18s} {'Raw':>10s}  {'Affine':>10s}")
print(f"  MAPE (all):       {mape_raw:>9.2f}%  {mape_affine:>9.2f}%")
print(f"  MAPE (last {tail_n}d): {mape_raw_tail:>9.2f}%  {mape_aff_tail:>9.2f}%")
print(f"  MAE  (all):       {mae_raw:>9.0f}$  {mae_affine:>9.0f}$")
print("=" * 60)

# --- Plot ---
dates = train_data.index[window_size:]

fig, axes = plt.subplots(2, 1, figsize=(20, 12), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})

# Panel 1: giá thực tế vs raw vs affine
ax = axes[0]
ax.plot(dates, y_true, label="Thực tế", color="blue", linewidth=1.2)
ax.plot(dates, preds_inverse, label=f"Raw  (MAPE {mape_raw:.2f}%)",
        color="orange", alpha=0.6, linewidth=0.8)
ax.plot(dates[min_cal:], preds_affine[min_cal:],
        label=f"Affine (MAPE {mape_affine:.2f}%)",
        color="green", linestyle="--", linewidth=1.0)
ax.set_title("Backtest – Raw vs Rolling Affine Calibration")
ax.set_ylabel("Giá (USD)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)

# Panel 2: % error
pct_err_raw    = (preds_inverse - y_true) / y_true * 100
pct_err_affine = (preds_affine[min_cal:] - y_true[min_cal:]) / y_true[min_cal:] * 100

ax2 = axes[1]
ax2.fill_between(dates, pct_err_raw, 0, alpha=0.3, color="orange", label="Raw % error")
ax2.plot(dates[min_cal:], pct_err_affine, color="green", linewidth=0.8,
         label="Affine % error")
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_ylabel("% Error")
ax2.set_xlabel("Ngày")
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from collections import deque
from tqdm.auto import tqdm

# ==== Forecast Next N days (V2 – Affine + Anchor) ========================

# 1) Re-scale TOÀN BỘ dữ liệu thực tế (train + test) bằng cùng scaler
all_scaled_values = scaler.transform(df[features].values)   # shape (len(df), n_feat)

# 2) Fit Affine Calibration trên TOÀN BỘ dữ liệu thực tế
#    Dự đoán backtest trên toàn bộ data để tìm (a, b)
all_X = []
for i in range(window_size, len(all_scaled_values)):
    all_X.append(all_scaled_values[i - window_size:i].flatten())
all_X = np.asarray(all_X).reshape(-1, 1, window_size * num_features)

all_preds_scaled = model.predict(all_X, verbose=0)
all_preds_inv = scaler.inverse_transform(
    np.c_[all_preds_scaled, np.zeros((len(all_preds_scaled), num_features - 1))]
)[:, 0]

target_col = features[0]
all_y_true = df[target_col].values[window_size:]

# Fit affine trên cal_window ngày cuối cùng (gần nhất → chính xác nhất)
cal_window = 120
a_final, b_final = np.polyfit(
    all_preds_inv[-cal_window:], all_y_true[-cal_window:], 1
)
print(f"[INFO] Affine calibration (last {cal_window}d): a={a_final:.6f}, b={b_final:.2f}")
print(f"[INFO] Ngày thực tế cuối cùng: {df.index[-1].strftime('%Y-%m-%d')}")
print(f"[INFO] Giá {target_col} cuối cùng: {df[target_col].iloc[-1]:,.2f} USD")

# 3) Hàm forecast cải tiến
def forecast_n_days_v2(model, scaled_data, scaler, features, window_size, n_days,
                       last_date, last_price, affine_a, affine_b):
    """
    Dự đoán n_days ngày, bắt đầu từ ngày thực tế cuối cùng.
    Cải tiến:
      - Seed = window_size ngày thực tế cuối cùng (đã scaled)
      - Áp dụng Affine Calibration (a, b) từ backtest
      - Neo (anchor) dự đoán sát giá thực tế cuối cùng, giảm dần
      - Cập nhật day_of_week cho từng ngày tương lai
    """
    n_feat = len(features)
    window = deque(scaled_data[-window_size:], maxlen=window_size)
    forecasts_scaled = np.empty(n_days, dtype="float32")

    # Index của day_of_week trong features (nếu có)
    dow_idx = features.index('day_of_week') if 'day_of_week' in features else None

    for i in tqdm(range(n_days), desc="Forecasting"):
        inp = np.array(window).reshape(1, 1, -1)
        pred = model.predict(inp, verbose=0)[0, 0]
        forecasts_scaled[i] = pred

        # Tạo vector feature mới
        last_row = list(window[-1])  # copy
        last_row[0] = pred           # cập nhật cột target

        # Cập nhật day_of_week cho ngày tương lai
        if dow_idx is not None:
            future_date = last_date + pd.Timedelta(days=i + 1)
            raw_dow = future_date.dayofweek / 6.0
            dummy = np.zeros((1, n_feat))
            dummy[0, dow_idx] = raw_dow
            last_row[dow_idx] = scaler.transform(dummy)[0, dow_idx]

        window.append(np.array(last_row))

    # ---- Inverse transform về USD ----
    raw_inv = scaler.inverse_transform(
        np.c_[forecasts_scaled, np.zeros((n_days, n_feat - 1))]
    )[:, 0]

    # ---- Áp dụng Affine Calibration ----
    calibrated = affine_a * raw_inv + affine_b

    # ---- Anchor: neo giá ngày đầu tiên sát giá thực tế cuối cùng ----
    # Offset giảm dần theo hàm mũ (half-life ~30 ngày)
    offset = last_price - calibrated[0]
    decay = np.exp(-np.arange(n_days) / 30.0)
    anchored = calibrated + offset * decay

    future_dates = pd.date_range(last_date + pd.Timedelta(days=1),
                                 periods=n_days, freq="D")
    return pd.Series(anchored, index=future_dates, name="Forecast")


# ------------------------------ RUN ---------------------------------------
last_actual_date  = df.index[-1]
last_actual_price = df[target_col].iloc[-1]

forecast_90 = forecast_n_days_v2(
    model,
    all_scaled_values,
    scaler,
    features,
    window_size,
    n_days=90,
    last_date=last_actual_date,
    last_price=last_actual_price,
    affine_a=a_final,
    affine_b=b_final,
)

# ---- In dự đoán từng ngày -----------------------------------------------
forecast_table = forecast_90.to_frame(name="Forecast_USD")
forecast_table.index.name = "Date"
print(f"\n===== Forecast chi tiết 90 ngày tới (bắt đầu từ {last_actual_date.strftime('%Y-%m-%d')}) =====")
print(forecast_table.round(2))
print("\n===== Dự đoán từng ngày =====")
for date, value in forecast_90.items():
    print(f"{date:%Y-%m-%d}: {value:,.2f} USD")

# ---- Vẽ biểu đồ kết hợp: dữ liệu thực tế + forecast -------------------
tail_actual = 90   # Hiển thị 90 ngày thực tế gần nhất

fig, ax = plt.subplots(figsize=(20, 7))

# Dữ liệu thực tế (đuôi)
actual_tail = df[target_col].iloc[-tail_actual:]
ax.plot(actual_tail.index, actual_tail.values,
        label=f"Thực tế ({target_col})", color="blue", linewidth=1.5)

# Đoạn nối liền mạch: ngày cuối thực tế → ngày đầu forecast
bridge = pd.Series([last_actual_price, forecast_90.iloc[0]],
                   index=[last_actual_date, forecast_90.index[0]])
ax.plot(bridge.index, bridge.values, color="orange", linewidth=1.5)

# Forecast 90 ngày
ax.plot(forecast_90.index, forecast_90.values,
        label="Forecast 90 ngày (Affine + Anchor)", color="orange",
        linewidth=1.5, linestyle="--")

# Confidence band ±5%
ax.fill_between(forecast_90.index,
                forecast_90.values * 0.95,
                forecast_90.values * 1.05,
                color="orange", alpha=0.15, label="Khoảng tin cậy ±5%")

# Đường thẳng ngang = giá cuối cùng
ax.axhline(last_actual_price, color="gray", linewidth=0.7, linestyle=":")
ax.axvline(last_actual_date, color="red", linewidth=0.8, linestyle="--",
           label=f"Ngày cuối thực tế ({last_actual_date.strftime('%Y-%m-%d')})")

ax.set_title(f"Gold {target_col} – Forecast 90 ngày (Affine Calibrated + Anchored)")
ax.set_xlabel("Ngày")
ax.set_ylabel("Giá (USD)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# # Hàm dự đoán và vẽ biểu đồ
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from matplotlib.dates import DateFormatter
# from datetime import timedelta
# import io
# from PIL import Image

# def predict_and_plot(days_to_predict, df, df_scaled, scaler, features, model):
#     window_size = 365
#     num_features = len(features)

#     if len(df_scaled) < window_size:
#         raise ValueError(f"Không đủ dữ liệu: cần ít nhất {window_size} ngày, chỉ có {len(df_scaled)} ngày.")

#     seed = df_scaled.values[-window_size:].flatten().reshape(1, 1, window_size * num_features)
#     predictions_scaled = []
    
#     for _ in range(days_to_predict):
#         pred = model.predict(seed, verbose=0)[0, 0]
#         predictions_scaled.append(pred)
#         next_row = seed[0, 0, -num_features:].copy()
#         next_row[0] = pred
#         seed = np.append(seed[0, 0, num_features:], next_row).reshape(1, 1, window_size * num_features)

#     predictions = scaler.inverse_transform(
#         np.hstack([np.array(predictions_scaled).reshape(-1, 1), np.zeros((days_to_predict, num_features-1))])
#     )[:, 0]

#     # Thêm khoảng tin cậy (±5%)
#     ci_percentage = 0.05  # 5%
#     lower_bound = predictions * (1 - ci_percentage)
#     upper_bound = predictions * (1 + ci_percentage)

#     arr_pred_index = pd.date_range(start=df.index[-1] + timedelta(days=1), periods=days_to_predict, freq='D')
#     forecast_df = pd.DataFrame({
#         'Forecast': predictions,
#         'Lower_Bound': lower_bound,
#         'Upper_Bound': upper_bound
#     }, index=arr_pred_index)

#     result_text = "               Dự đoán / Forecast\n"
#     for date, row in forecast_df.iterrows():
#         result_text += f"{date.strftime('%Y-%m-%d')}  {row['Forecast']:.2f} USD (Khoảng tin cậy: {row['Lower_Bound']:.2f} - {row['Upper_Bound']:.2f} USD) / {date.strftime('%Y-%m-%d')}  {row['Forecast']:.2f} USD (CI: {row['Lower_Bound']:.2f} - {row['Upper_Bound']:.2f} USD)\n"
#     result_text = result_text.rstrip('\n')

#     # Biểu đồ dự đoán với nhãn song ngữ
#     plt.figure(figsize=(15, 5))
#     plt.plot(forecast_df['Forecast'], label='Dự đoán / Forecast (Gold)', marker='o', markersize=5, color='orange')
#     plt.fill_between(forecast_df.index, forecast_df['Lower_Bound'], forecast_df['Upper_Bound'], 
#                      color='orange', alpha=0.3, label='Khoảng tin cậy (±5%) / Confidence Interval (±5%)')
#     plt.title(f"Dự đoán giá Vàng cho {len(forecast_df)} ngày tới / Gold Price Forecast for the Next {len(forecast_df)} Days")
#     plt.xlabel("Ngày / Date")
#     plt.ylabel("Giá (USD) / Price (USD)")
#     plt.legend()
#     plt.grid(True)
#     plt.xticks(rotation=45)
#     plt.gca().xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', bbox_inches='tight')
#     buf.seek(0)
#     forecast_img = Image.open(buf)
#     plt.close()

#     # Biểu đồ so sánh giá thực tế và dự đoán với nhãn song ngữ
#     true_values = df['Close']
#     plt.figure(figsize=(15, 5))
#     plt.plot(true_values, label='Giá thực tế / Actual Price (Gold)', color='blue')
#     plt.plot(forecast_df['Forecast'], label='Dự đoán / Forecast (Gold)', color='orange')
#     plt.fill_between(forecast_df.index, forecast_df['Lower_Bound'], forecast_df['Upper_Bound'], 
#                      color='orange', alpha=0.3, label='Khoảng tin cậy (±5%) / Confidence Interval (±5%)')
#     plt.title("Giá Vàng thực tế và dự đoán / Actual and Predicted Gold Prices")
#     plt.xlabel("Ngày / Date")
#     plt.ylabel("Giá (USD) / Price (USD)")
#     plt.legend()
#     plt.grid(True)
#     plt.xticks(rotation=45)
#     plt.gca().xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', bbox_inches='tight')
#     buf.seek(0)
#     combined_img = Image.open(buf)
#     plt.close()

#     return result_text, forecast_img, combined_img

In [ ]:
# # ==== Precompute Predictions for 1 to 30 Days ====
# # Tính trước dự đoán từ 1 đến 30 ngày
# import os
# os.makedirs('forecast_images', exist_ok=True)
# results = {}
# for days in range(1, 31):
#     result_text, forecast_img, combined_img = predict_and_plot(days, df, df_scaled, scaler, features, model)
#     if result_text and forecast_img and combined_img:
#         results[days] = {
#             'result_text': result_text,
#             'forecast_img': forecast_img,
#             'combined_img': combined_img,
#             'forecast_img_path': os.path.join('forecast_images', f'forecast_{days}.png'),
#             'combined_img_path': os.path.join('forecast_images', f'combined_{days}.png')
#         }
#     else:
#         print(f"Không thể tạo kết quả cho {days} ngày.")

# # ==== Lưu hình ảnh vào đĩa ====
# for days, data in results.items():
#     try:
#         data['forecast_img'].save(data['forecast_img_path'])
#         data['combined_img'].save(data['combined_img_path'])
#         print(f"Đã lưu ảnh cho {days} ngày: {data['forecast_img_path']} và {data['combined_img_path']}")
#     except Exception as e:
#         print(f"Lỗi lưu ảnh cho {days} ngày: {str(e)}")

# # ==== Zip the Forecast Images ====
# folder_to_zip = 'forecast_images'
# zip_output = 'forecast_images.zip'

# with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zipf:
#     for root, dirs, files in os.walk(folder_to_zip):
#         for file in files:
#             file_path = os.path.join(root, file)
#             zipf.write(file_path, os.path.relpath(file_path, '.'))
#             print(f"Đã thêm file vào zip: {file}")

# print(f"Created zip file: {zip_output}")

In [ ]:
# #Lưu kết quả vào CSV
# df_results = pd.DataFrame.from_dict({k: v['result_text'] for k, v in results.items()}, orient='index', columns=['Forecast'])
# df_results.to_csv('precomputed_predictions.csv', index_label='Days')

In [ ]:
# # Lưu kết quả vào CSV với Confidence Interval ±5% cho mỗi dự đoán
# import re

# def parse_forecast_text(text):
#     """
#     Parse chuỗi result_text để trích xuất Date, Forecast, Lower_Bound, Upper_Bound.
#     Confidence Interval được tính ±5% của Forecast.
#     """
#     pattern = r"(\d{4}-\d{2}-\d{2})  (\d+\.\d+) USD \(Khoảng tin cậy: (\d+\.\d+) - (\d+\.\d+) USD\) /"
#     matches = re.findall(pattern, text)
#     data = []
#     for match in matches:
#         date, forecast, lower, upper = match
#         forecast_val = float(forecast)
#         lower_val = float(lower)
#         upper_val = float(upper)
#         # Xác nhận CI là ±5%
#         assert abs(lower_val - forecast_val * 0.95) < 1e-2, "Lower bound không khớp ±5%"
#         assert abs(upper_val - forecast_val * 1.05) < 1e-2, "Upper bound không khớp ±5%"
#         data.append({'Date': date, 'Forecast': forecast_val, 'Lower_Bound': lower_val, 'Upper_Bound': upper_val})
#     return pd.DataFrame(data)

# all_data = []
# for days, data in results.items():
#     forecast_df = parse_forecast_text(data['result_text'])
#     all_data.append(forecast_df)

# df_all = pd.concat(all_data)
# df_all.to_csv('precomputed_predictions.csv', index=False)

# print("Đã lưu file 'precomputed_predictions.csv' với các cột: Date, Forecast, Lower_Bound (±5%), Upper_Bound (±5%).")